# Install Required Libraries
In this cell, we install all libraries required for building the retrieval pipeline.
It includes FAISS for vector indexing, HuggingFace Transformers for FashionCLIP and BLIP models, and supporting utilities.

In [ ]:
!pip install -q transformers faiss-cpu torch torchvision pillow tqdm matplotlib

# Import Packages
This cell imports all the Python libraries used in the project such as FAISS, PyTorch, Transformers, PIL, and numpy.
These will be used for feature extraction, indexing, metadata storage, and retrieval.

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
import os, pickle, random, zipfile, re
import numpy as np
from tqdm import tqdm
from PIL import Image
import cv2

import torch
import faiss
from transformers import CLIPProcessor, CLIPModel
from transformers import BlipProcessor, BlipForConditionalGeneration, BlipForImageTextRetrieval


# Load FashionCLIP for Embedding Extraction
This cell loads FashionCLIP, a fashion-domain aligned CLIP model.
It converts both images and text queries into embeddings in the same vector space, enabling semantic retrieval.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

FASHIONCLIP_NAME = "patrickjohncyh/fashion-clip"  # FashionCLIP
clip_model = CLIPModel.from_pretrained(FASHIONCLIP_NAME).to(device)
clip_processor = CLIPProcessor.from_pretrained(FASHIONCLIP_NAME)

print("✅ FashionCLIP loaded")


# Load BLIP Captioning and Cross-Encoder Models
BLIP Captioning is used to generate image descriptions which help extract attributes like clothing type, color and context.

In [ ]:
BLIP_CAPTION_NAME = "Salesforce/blip-image-captioning-base"
blip_cap_model = BlipForConditionalGeneration.from_pretrained(BLIP_CAPTION_NAME).to(device)
blip_cap_processor = BlipProcessor.from_pretrained(BLIP_CAPTION_NAME)

print("✅ BLIP caption model loaded")


BLIP ITM (Image-Text Matching) is used as a cross-encoder reranker to boost final retrieval quality.

In [ ]:
BLIP_ITM_NAME = "Salesforce/blip-itm-base-coco"
itm_model = BlipForImageTextRetrieval.from_pretrained(BLIP_ITM_NAME).to(device)
itm_processor = BlipProcessor.from_pretrained(BLIP_ITM_NAME)

print("✅ BLIP ITM cross-encoder loaded")


# Dataset Setup and Extraction

In [ ]:
zip_path = "/content/Glance_Train1.zip"
extract_path = "/content/dataset/train"
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Extracted to:", extract_path)


In [ ]:
zip_path = "/content/Glance_Train2.zip"
extract_path = "/content/dataset/train"
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Extracted to:", extract_path)


In [ ]:
zip_path = "/content/Glance_Train3.zip"
extract_path = "/content/dataset/train3"
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Extracted to:", extract_path)


In [ ]:
zip_path = "/content/Glance_Train4.zip"
extract_path = "/content/dataset/train4"
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Extracted to:", extract_path)


# Collect All Image Paths from Dataset Directory

In [ ]:
def get_all_images(folder):
    valid = (".jpg", ".jpeg", ".png", ".webp")
    paths = []
    for root, _, files in os.walk(folder):
        for f in files:
            if f.lower().endswith(valid):
                paths.append(os.path.join(root, f))
    return paths


# Region-Based Color Extraction (Top/Bottom Matching)

In [ ]:
def dominant_color_name(rgb):
    # Very simple mapping (enough for internship)
    r, g, b = rgb
    if r > 160 and g < 120 and b < 120: return "red"
    if b > 160 and r < 120 and g < 160: return "blue"
    if g > 160 and r < 140 and b < 140: return "green"
    if r > 180 and g > 180 and b > 180: return "white"
    if r < 60 and g < 60 and b < 60: return "black"
    if r > 160 and g > 140 and b < 120: return "yellow"
    return "unknown"

def region_colors(pil_img: Image.Image):
    img = np.array(pil_img)
    h = img.shape[0]

    top = img[:h//2, :, :]
    bottom = img[h//2:, :, :]

    def mean_rgb(region):
        mean = region.reshape(-1, 3).mean(axis=0)
        return tuple(mean.astype(int))

    top_color = dominant_color_name(mean_rgb(top))
    bot_color = dominant_color_name(mean_rgb(bottom))
    return {"top_color": top_color, "bottom_color": bot_color}


# Define Fashion Vocabulary & Attribute Extraction
In this cell, we define the vocabulary of key fashion entities such as colors, clothing types (topwear, bottomwear, one-piece) and accessories.

In [ ]:
COLORS = ["red","blue","green","black","white","yellow","pink","orange","purple","brown","grey","gray","beige"]

TOP_CLOTHES = sorted(set([
    "shirt", "tshirt", "t-shirt", "tee", "polo", "blouse", "top", "tank top", "crop top",
    "camisole", "sweatshirt", "hoodie", "sweater", "cardigan",
    "jacket", "blazer", "coat", "raincoat", "overcoat",
    "kurti", "kurta", "tunic"
]))

BOTTOM_CLOTHES = sorted(set([
    "pants", "jeans", "denim", "trousers", "chinos", "shorts", "skirt",
    "leggings", "joggers", "track pants",
    "salwar", "churidar", "palazzo"
]))

ONE_PIECE = sorted(set([
    "dress", "gown", "frock", "maxi", "midi dress", "minidress", "shift dress", "bodycon",
    "jumpsuit", "romper", "playsuit",
    "saree", "sari", "lehenga", "anarkali", "kameez", "salwar suit",
    "onesie", "overall", "overalls", "dungaree", "dungarees"
]))

ACCESSORIES = sorted(set([
    "tie", "belt", "scarf", "cap", "hat", "sunglasses", "watch", "shoes", "sneakers"
]))


SCENES = ["street","beach","office","wedding","park","mall","indoor","outdoor"]

ALL_CLOTHES = sorted(set(TOP_CLOTHES + BOTTOM_CLOTHES + ONE_PIECE + ACCESSORIES))


def extract_attributes(text):
    text = text.lower()

    attrs = {"colors": [], "clothes": [], "scene": []}

    # colors
    for c in COLORS:
        if c in text:
            attrs["colors"].append(c)

    # clothes (now covers ALL categories)
    for t in ALL_CLOTHES:
        if t in text:
            attrs["clothes"].append(t)

    # scenes
    for s in SCENES:
        if s in text:
            attrs["scene"].append(s)

    return attrs


# Extract Prebuilt Index Files from ZIP

In [ ]:
import os, zipfile, pickle
import faiss

# ✅ Path to your zip
ZIP_INDEX_PATH = "/content/index_folder.zip"

# ✅ Where to extract it
EXTRACT_DIR = "/content/index_merged"
os.makedirs(EXTRACT_DIR, exist_ok=True)

# ✅ Unzip index files
with zipfile.ZipFile(ZIP_INDEX_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("✅ Index zip extracted to:", EXTRACT_DIR)
print("Extracted files:", os.listdir(EXTRACT_DIR))


# Load Merged FAISS Index and Initialize Retrieval Models
This cell loads the final merged FAISS index along with metadata files such as image paths, attributes, and region-based color information.

In [ ]:

INDEX_DIR = "/content/index_merged"
index = faiss.read_index(os.path.join(INDEX_DIR, "faiss_index.bin"))

with open(os.path.join(INDEX_DIR, "image_paths.pkl"), "rb") as f:
    image_paths = pickle.load(f)
with open(os.path.join(INDEX_DIR, "attributes.pkl"), "rb") as f:
    attributes = pickle.load(f)
with open(os.path.join(INDEX_DIR, "region_meta.pkl"), "rb") as f:
    region_meta = pickle.load(f)

# FashionCLIP
FASHIONCLIP_NAME = "patrickjohncyh/fashion-clip"
clip_model = CLIPModel.from_pretrained(FASHIONCLIP_NAME).to(device)
clip_processor = CLIPProcessor.from_pretrained(FASHIONCLIP_NAME)

# Cross encoder (BLIP ITM)
BLIP_ITM_NAME = "Salesforce/blip-itm-base-coco"
itm_model = BlipForImageTextRetrieval.from_pretrained(BLIP_ITM_NAME).to(device)
itm_processor = BlipProcessor.from_pretrained(BLIP_ITM_NAME)

print("✅ Retriever ready. Images:", len(image_paths))


# Convert Query Text to FashionCLIP Embedding

In [ ]:
def clip_text_embedding(text: str):
    inputs = clip_processor(text=[text], return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        feat = clip_model.get_text_features(**inputs)
    feat = feat / feat.norm(dim=-1, keepdim=True)
    return feat.cpu().numpy().astype("float32")


# Attribute Matching Score (Color + Clothing + Scene)
This function computes a relevance score by comparing query attributes with image attributes extracted from BLIP captions.

In [ ]:
def attribute_match_score(query_attrs, img_attrs):
    score = 0
    score += len(set(query_attrs["colors"]) & set(img_attrs["colors"])) * 2
    score += len(set(query_attrs["clothes"]) & set(img_attrs["clothes"])) * 3
    score += len(set(query_attrs["scene"]) & set(img_attrs["scene"])) * 1
    return score

def region_match_score(query_text, reg_info):
    q = query_text.lower()
    score = 0
    if "shirt" in q or "top" in q or "blazer" in q:
        for c in COLORS:
            if c in q and reg_info.get("top_color") == c:
                score += 2
    if "pants" in q or "jeans" in q or "skirt" in q:
        for c in COLORS:
            if c in q and reg_info.get("bottom_color") == c:
                score += 2
    return score


# Cross-Encoder Reranker using BLIP Image-Text Matching

In [ ]:
import torch.nn.functional as F

def itm_score(query_text: str, image_path: str):
    img = Image.open(image_path).convert("RGB")

    inputs = itm_processor(images=img, text=query_text, return_tensors="pt", padding=True).to(device)

    with torch.no_grad():
        outputs = itm_model(**inputs)

        # outputs.itm_score has 2 logits: [not_match, match]
        logits = outputs.itm_score  # shape: [1, 2]

        probs = F.softmax(logits, dim=1)  # convert logits -> probability
        match_prob = probs[0, 1].item()   # take probability of "match"

    return float(match_prob)


# Final Retrieval Pipeline (FAISS → Rerank → Cross-Encoder)
This function performs end-to-end retrieval: it first uses FAISS to fetch top-N candidates using FashionCLIP similarity.
Then it applies attribute-based and region-based reranking, followed by BLIP-ITM reranking on the shortlisted images to return the best results.

In [ ]:
def search_best(query, top_k=8, topN=200, rerankN=50, alpha=0.65, beta=0.25, gamma=0.10):
    q_vec = clip_text_embedding(query)
    clip_scores, idxs = index.search(q_vec, topN)

    q_attrs = extract_attributes(query)
    candidates = []

    for base_score, idx in zip(clip_scores[0], idxs[0]):
        path = image_paths[idx]
        img_attrs = attributes.get(path, {"colors":[], "clothes":[], "scene":[]})
        reg = region_meta.get(path, {})

        attr_score = attribute_match_score(q_attrs, img_attrs)
        reg_score = region_match_score(query, reg)

        # hybrid rerank score (fast)
        fast_score = alpha * float(base_score) + beta * float(attr_score) + gamma * float(reg_score)
        candidates.append((path, fast_score, float(base_score), float(attr_score), float(reg_score)))

    # sort + take top rerankN for cross-encoder
    candidates.sort(key=lambda x: x[1], reverse=True)
    shortlist = candidates[:rerankN]

    final = []
    for (path, fast_score, c, a, r) in shortlist:
        ce = itm_score(query, path)  # cross encoder
        final_score = fast_score + 0.25 * ce  # add cross encoder influence
        final.append((path, final_score, c, a, r, ce))

    final.sort(key=lambda x: x[1], reverse=True)
    return final[:top_k], q_attrs


# Visualize Top Retrieved Images
This function displays the final top-k images returned by the retriever for the given query.

In [ ]:
def show_results(query):
    results, q_attrs = search_best(query, top_k=15)

    print("Query:", query)
    print("Extracted Query Attributes:", q_attrs)

    plt.figure(figsize=(20, 12))  # bigger for 15 images

    for i, (path, F, C, A, R, CE) in enumerate(results):
        img = Image.open(path).convert("RGB")

        plt.subplot(3, 5, i + 1)  # ✅ 15 images = 3 rows x 5 cols
        plt.imshow(img)
        plt.axis("off")
        plt.title(
            f"F:{F:.2f}\nC:{C:.2f} A:{A:.1f}\nR:{R:.1f} CE:{CE:.2f}",
            fontsize=9
        )

    plt.tight_layout()
    plt.show()

show_results("Professional business attire inside a modern office")


# More Test Cases

In [ ]:
def show_results(query):
    results, q_attrs = search_best(query, top_k=15)

    print("Query:", query)
    print("Extracted Query Attributes:", q_attrs)

    plt.figure(figsize=(20, 12))  # bigger for 15 images

    for i, (path, F, C, A, R, CE) in enumerate(results):
        img = Image.open(path).convert("RGB")

        plt.subplot(3, 5, i + 1)  # ✅ 15 images = 3 rows x 5 cols
        plt.imshow(img)
        plt.axis("off")
        plt.title(
            f"F:{F:.2f}\nC:{C:.2f} A:{A:.1f}\nR:{R:.1f} CE:{CE:.2f}",
            fontsize=9
        )

    plt.tight_layout()
    plt.show()

show_results("A person in a bright yellow raincoat")


In [ ]:
def show_results(query):
    results, q_attrs = search_best(query, top_k=15)

    print("Query:", query)
    print("Extracted Query Attributes:", q_attrs)

    plt.figure(figsize=(20, 12))  # bigger for 15 images

    for i, (path, F, C, A, R, CE) in enumerate(results):
        img = Image.open(path).convert("RGB")

        plt.subplot(3, 5, i + 1)  # ✅ 15 images = 3 rows x 5 cols
        plt.imshow(img)
        plt.axis("off")
        plt.title(
            f"F:{F:.2f}\nC:{C:.2f} A:{A:.1f}\nR:{R:.1f} CE:{CE:.2f}",
            fontsize=9
        )

    plt.tight_layout()
    plt.show()

show_results("Casual weekend outfit for a city walk")


In [ ]:
def show_results(query):
    results, q_attrs = search_best(query, top_k=15)

    print("Query:", query)
    print("Extracted Query Attributes:", q_attrs)

    plt.figure(figsize=(20, 12))  # bigger for 15 images

    for i, (path, F, C, A, R, CE) in enumerate(results):
        img = Image.open(path).convert("RGB")

        plt.subplot(3, 5, i + 1)  # ✅ 15 images = 3 rows x 5 cols
        plt.imshow(img)
        plt.axis("off")
        plt.title(
            f"F:{F:.2f}\nC:{C:.2f} A:{A:.1f}\nR:{R:.1f} CE:{CE:.2f}",
            fontsize=9
        )

    plt.tight_layout()
    plt.show()

show_results("Someone wearing a blue shirt sitting on a park bench")


In [ ]:
def show_results(query):
    results, q_attrs = search_best(query, top_k=15)

    print("Query:", query)
    print("Extracted Query Attributes:", q_attrs)

    plt.figure(figsize=(20, 12))  # bigger for 15 images

    for i, (path, F, C, A, R, CE) in enumerate(results):
        img = Image.open(path).convert("RGB")

        plt.subplot(3, 5, i + 1)  # ✅ 15 images = 3 rows x 5 cols
        plt.imshow(img)
        plt.axis("off")
        plt.title(
            f"F:{F:.2f}\nC:{C:.2f} A:{A:.1f}\nR:{R:.1f} CE:{CE:.2f}",
            fontsize=9
        )

    plt.tight_layout()
    plt.show()

show_results("A red tie and a white shirt in a formal settingt")
